# 22. Exercise Definition Test (Pipeline 03)

- Goal: check exercise-definition loading, preset expansion, validation warnings, and fallback behavior.
- Docs: `docs_eng/pipeline/03_exercise_definition.md` / `docs/pipeline/03_exercise_definition.md`
- Inputs: Canonical exercise definition YAML files, optional authoring draft bundle, and the default annotated sample context.
- Outputs: In-memory dataframes/reports unless a cell explicitly saves under `data/processed/`.
- Checks: Loader reports, expected fallback paths, and pipeline report integration.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

import warnings

from movement.exercise_definition import load_exercise_definition

DEFINITIONS_DIR = PROJECT_ROOT / "data/definitions/exercises"
AUTHORING_EXAMPLE_ROOT = PROJECT_ROOT / "data/examples/exercise_authoring"
AUTHORING_DRAFT_ROOT = PROJECT_ROOT / "data/processed/authoring_drafts"

# Default stage-check target. Change to "draft_squat" to test an authoring draft.
TARGET_EXERCISE_ID = "squat"


def candidate_definition_dirs(exercise_id):
    return [
        DEFINITIONS_DIR,
        AUTHORING_EXAMPLE_ROOT / exercise_id / "data/definitions/exercises",
        AUTHORING_DRAFT_ROOT / exercise_id / "data/definitions/exercises",
    ]


def resolve_target_definitions_dir(exercise_id):
    for definitions_dir in candidate_definition_dirs(exercise_id):
        if (definitions_dir / f"{exercise_id}.yaml").exists():
            return definitions_dir
    return DEFINITIONS_DIR


TARGET_DEFINITIONS_DIR = resolve_target_definitions_dir(TARGET_EXERCISE_ID)


## Case 1: Load Selected Definition

For the current p01 stage-check path, load one selected definition first.
The default target is `squat` because the sample annotation uses `exercise_type: squat`.
If `TARGET_EXERCISE_ID` is changed to an authoring draft such as `draft_squat`,
the notebook resolves the matching example or local draft bundle before falling back to the canonical directory.
Broader registry coverage belongs in unit tests or a separate registry audit.

In [ ]:
exercise_def = load_exercise_definition(TARGET_EXERCISE_ID, TARGET_DEFINITIONS_DIR)
selected_defs = {exercise_def.exercise_id: exercise_def}

print("loaded selected definition:")
print(f"  exercise_id     : {exercise_def.exercise_id}")
print(f"  version         : {exercise_def.version}")
print(f"  fallback        : {exercise_def.is_generic_fallback}")
print(f"  definitions_dir : {TARGET_DEFINITIONS_DIR}")

In [ ]:
import pandas as pd

expected_path = TARGET_DEFINITIONS_DIR / f"{TARGET_EXERCISE_ID}.yaml"
assert expected_path.exists(), f"missing selected definition YAML: {expected_path}"
assert exercise_def.exercise_id == TARGET_EXERCISE_ID
assert exercise_def.is_generic_fallback is False

summary = pd.DataFrame([
    {
        'exercise_id': exercise_def.exercise_id,
        'display_name': exercise_def.display_name,
        'definitions_dir': str(TARGET_DEFINITIONS_DIR),
        'is_generic_fallback': exercise_def.is_generic_fallback,
        'laterality': exercise_def.classification.get('laterality'),
        'posture_type': exercise_def.classification.get('posture_type'),
    }
])
display(summary)
print("PASS: selected exercise definition loaded for the current stage-check target")


## Case 2: Selected Definition Field Inspection

Spot-check the most important typed fields for the selected definition.

In [ ]:
for ex_id, ed in selected_defs.items():
    clf = ed.classification
    print(f"── {ex_id} ──────────────────────────────")
    print(f"  laterality       : {clf.get('laterality')}")
    print(f"  posture_type     : {clf.get('posture_type')}")
    print(f"  primary_plane    : {clf.get('primary_plane')}")
    print(f"  phase_model.type : {ed.phase_model.type}")
    print(f"  expected_ratio   : {ed.phase_model.expected_ratio}")
    print(f"  primary_joints   : {ed.landmarks.primary_joints}")
    print(f"  compensation_candidates ({len(ed.compensation_candidates)}): {ed.compensation_candidates}")
    print()

## Case 3: Required Field Validation

A YAML missing a required field must raise `ValueError`.

In [ ]:
import tempfile
import yaml

# Write a minimal YAML missing the 'landmarks' field
bad_yaml = {
    "exercise_id": "test_incomplete",
    "classification": {"family": "lower_body", "laterality": "bilateral_symmetric",
                       "posture_type": "standing", "kinetic_chain": "closed_chain",
                       "primary_plane": "sagittal"},
    "phase_model": {"type": "resistance_phase", "expected_ratio": {"eccentric": 0.5, "concentric": 0.5}},
    # 'landmarks' intentionally omitted
    "compensation_candidates": [],
    "feature_domains": {"spatial": [], "temporal": [], "control": [], "biomechanical_proxy": []},
    "quality_rules": {"minimum_visible_landmark_ratio": 0.8},
}

with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "test_incomplete.yaml"
    p.write_text(yaml.dump(bad_yaml), encoding="utf-8")
    # also copy generic fallback so the dir is valid
    import shutil
    shutil.copy(DEFINITIONS_DIR / "generic.yaml", Path(tmpdir) / "generic.yaml")

    try:
        load_exercise_definition("test_incomplete", tmpdir)
        print("FAIL: expected ValueError was not raised")
    except (ValueError, FileNotFoundError) as e:
        print("PASS: definition load failed as expected for this negative fixture")
        print(f"  {e}")

## Case 4: Vocabulary Warning

An out-of-vocabulary value must emit a `UserWarning` but still load successfully.

In [ ]:
bad_vocab_yaml = {
    "exercise_id": "test_vocab",
    "classification": {
        "family": "lower_body",
        "laterality": "bilateral_symmetric",
        "posture_type": "INVALID_POSTURE",  # out-of-vocabulary
        "kinetic_chain": "closed_chain",
        "primary_plane": "sagittal",
    },
    "phase_model": {"type": "resistance_phase",
                   "expected_ratio": {"eccentric": 0.5, "concentric": 0.5}},
    "landmarks": {"model": "mediapipe_pose_33", "primary_joints": ["left_hip"],
                  "critical_landmarks": [23]},
    "compensation_candidates": [],
    "feature_domains": {"spatial": [], "temporal": [], "control": [], "biomechanical_proxy": []},
    "quality_rules": {"minimum_visible_landmark_ratio": 0.8},
}

with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "test_vocab.yaml"
    p.write_text(yaml.dump(bad_vocab_yaml), encoding="utf-8")
    shutil.copy(DEFINITIONS_DIR / "generic.yaml", Path(tmpdir) / "generic.yaml")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        ed = load_exercise_definition("test_vocab", tmpdir)

    vocab_warns = [w for w in caught if "not in controlled vocabulary" in str(w.message)]
    print(f"PASS: loaded successfully with {len(vocab_warns)} vocabulary warning(s)")
    for w in vocab_warns:
        print(f"  {w.message}")

## Case 5: Phase Ratio Warning

When `resistance_phase` or `task_phase` ratios do not sum to ≈ 1.0,
a `UserWarning` must be emitted.

In [ ]:
bad_ratio_yaml = {
    "exercise_id": "test_ratio",
    "classification": {"family": "lower_body", "laterality": "bilateral_symmetric",
                       "posture_type": "standing", "kinetic_chain": "closed_chain",
                       "primary_plane": "sagittal"},
    "phase_model": {
        "type": "resistance_phase",
        "expected_ratio": {"eccentric": 0.3, "isometric": 0.1, "concentric": 0.3},  # sum = 0.7
    },
    "landmarks": {"model": "mediapipe_pose_33", "primary_joints": ["left_hip"],
                  "critical_landmarks": [23]},
    "compensation_candidates": [],
    "feature_domains": {"spatial": [], "temporal": [], "control": [], "biomechanical_proxy": []},
    "quality_rules": {"minimum_visible_landmark_ratio": 0.8},
}

with tempfile.TemporaryDirectory() as tmpdir:
    p = Path(tmpdir) / "test_ratio.yaml"
    p.write_text(yaml.dump(bad_ratio_yaml), encoding="utf-8")
    shutil.copy(DEFINITIONS_DIR / "generic.yaml", Path(tmpdir) / "generic.yaml")

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        ed = load_exercise_definition("test_ratio", tmpdir)

    ratio_warns = [w for w in caught if "expected_ratio sums to" in str(w.message)]
    print(f"PASS: {len(ratio_warns)} phase ratio warning(s) emitted")
    for w in ratio_warns:
        print(f"  {w.message}")

## Case 6: Generic Fallback — None exercise_id

Passing `None` as `exercise_id` must load the generic definition and set
`is_generic_fallback=True`.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    generic_def = load_exercise_definition(None, DEFINITIONS_DIR)

fallback_warns = [w for w in caught if "generic fallback" in str(w.message).lower()]

assert generic_def.exercise_id == "generic"
assert generic_def.is_generic_fallback is True
assert len(fallback_warns) >= 1
print("PASS: None exercise_id → generic fallback with warning")
print(f"  warning: {fallback_warns[0].message}")

## Case 7: Generic Fallback — Missing File

An unknown `exercise_id` whose YAML does not exist must silently fall back
to generic and set `is_generic_fallback=True`.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    fallback_def = load_exercise_definition("nonexistent_exercise", DEFINITIONS_DIR)

fallback_warns = [w for w in caught if "not found" in str(w.message).lower()]

assert fallback_def.exercise_id == "generic"
assert fallback_def.is_generic_fallback is True
assert len(fallback_warns) >= 1
print("PASS: missing YAML → generic fallback with warning")
print(f"  warning: {fallback_warns[0].message}")

## Case 8: Pipeline Step Integration

Running the pipeline with `exercise_definition.enabled: true` must add
an `'exercise_definition'` key to the report dict.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

from movement.annotation import load_annotation_csv
from movement.config import LANDMARKS
from movement.io import load_pose_csv
from movement.pipeline import load_pipeline_config, run_pipeline

config_path = PROJECT_ROOT / "configs/pipeline_default.yaml"
csv_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ann_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv"

config = load_pipeline_config(config_path)
df = load_pose_csv(csv_path)
ann_df = load_annotation_csv(ann_path)

# Step 3 normally receives exercise context from Step 2 annotation when
# exercise_id is not explicitly configured. Enable annotation here so this
# stage check validates that handoff instead of the generic fallback path.
config.annotation.enabled = True
config.annotation.path = ann_path

selected_exercise_id = globals().get("TARGET_EXERCISE_ID", "squat")
selected_definitions_dir = globals().get(
    "TARGET_DEFINITIONS_DIR",
    PROJECT_ROOT / "data/definitions/exercises",
)

# If Case 1 was switched to a draft or another selected definition, run the
# pipeline against that explicit definition. With the default squat target,
# keep the annotation-derived handoff path under test.
if selected_exercise_id != "squat":
    config.exercise_definition.exercise_id = selected_exercise_id
    config.exercise_definition.definitions_dir = str(selected_definitions_dir)

print("annotation.enabled:", config.annotation.enabled)
print("annotation.path:", config.annotation.path)
print("selected_exercise_id:", selected_exercise_id)
print("selected_definitions_dir:", selected_definitions_dir)
print("exercise_definition.enabled:", config.exercise_definition.enabled)
print("exercise_definition.definitions_dir:", config.exercise_definition.definitions_dir)
print("exercise_definition.exercise_id:", config.exercise_definition.exercise_id)

In [ ]:
import warnings as _w

with _w.catch_warnings(record=True) as caught:
    _w.simplefilter("always")
    result_df, report = run_pipeline(df, config=config, landmarks=LANDMARKS, ann_df=ann_df)

print("steps executed:", list(report.keys()))
print()

if caught:
    print(f"{len(caught)} warning(s) during pipeline run:")
    for w in caught:
        print(f"  [{w.category.__name__}] {w.message}")

In [ ]:
import json

assert "exercise_definition" in report, "exercise_definition step missing from report"
exd_report = report["exercise_definition"]
print(json.dumps(exd_report, indent=2))

annotated_exercise_ids = sorted(
    str(value) for value in ann_df["exercise_type"].dropna().unique()
)
configured_exercise_id = config.exercise_definition.exercise_id
configured_definitions_dir = Path(config.exercise_definition.definitions_dir)
if not configured_definitions_dir.is_absolute():
    configured_definitions_dir = PROJECT_ROOT / configured_definitions_dir
known_definition_ids = {path.stem for path in configured_definitions_dir.glob("*.yaml")}

if configured_exercise_id:
    if configured_exercise_id in known_definition_ids:
        assert exd_report["exercise_id"] == configured_exercise_id
        if configured_exercise_id != "generic":
            assert exd_report["is_generic_fallback"] is False
    else:
        assert exd_report["exercise_id"] == "generic"
        assert exd_report["is_generic_fallback"] is True
else:
    expected_specific_ids = [
        exercise_id
        for exercise_id in annotated_exercise_ids
        if exercise_id in known_definition_ids
    ]
    if expected_specific_ids:
        assert exd_report["exercise_id"] in expected_specific_ids, (
            "exercise_definition did not load the annotated exercise_type: "
            f"annotated={expected_specific_ids}, loaded={exd_report['exercise_id']}"
        )
        assert exd_report["is_generic_fallback"] is False
    else:
        assert exd_report["is_generic_fallback"] is True

print()
print("PASS: exercise_definition step in report")
print(f"  configured exercise_id        : {configured_exercise_id}")
print(f"  configured definitions_dir    : {configured_definitions_dir}")
print(f"  annotated exercise_type values: {annotated_exercise_ids}")
print(f"  exercise_id                   : {exd_report['exercise_id']}")
print(f"  is_generic_fallback           : {exd_report['is_generic_fallback']}")

## Check Summary

This notebook cell is a compact execution/QC checkpoint. Use the pipeline document linked in the header for definitions, interpretation policy, and scope.
